<a href="https://colab.research.google.com/github/x1001000/Colab-Notebooks/blob/main/MM_SOP%E5%90%91%E9%87%8F%E8%B3%87%E6%96%99%E5%BA%AB%E7%AE%A1%E7%90%86.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 列出雲端硬碟SOP資料夾內全部文件

In [ ]:
file_paths = []
import os
if not os.path.exists('/content/drive'):
    print('請先在左側掛接雲端硬碟後，再執行一次')
else:
    for root, folders, files in os.walk('/content/drive/MyDrive/SOP'):
        for file in files:
            file_path = os.path.join(root, file)
            file_paths.append(file_path)
            print(file_path)

/content/drive/MyDrive/SOP/研究部/MM中國數據.docx
/content/drive/MyDrive/SOP/研究部/產業決策平台SOP.docx
/content/drive/MyDrive/SOP/研究部/月報檢查重點.docx
/content/drive/MyDrive/SOP/研究部/月報月曆 SOP.pdf
/content/drive/MyDrive/SOP/研究部/月報行情圖 SOP.pdf
/content/drive/MyDrive/SOP/研究部/快報_語音朗讀 SOP_2023.pdf
/content/drive/MyDrive/SOP/研究部/快報_圖文編輯 SOP_2025.pdf
/content/drive/MyDrive/SOP/研究部/時間軸上傳格式.pdf
/content/drive/MyDrive/SOP/研究部/短評文字SOP_20240718.pdf
/content/drive/MyDrive/SOP/研究部/每日短評SOP_2023.pdf
/content/drive/MyDrive/SOP/研究部/數據焦點_下周焦點圖片模板.pdf
/content/drive/MyDrive/SOP/研究部/數據焦點_下週焦點_2023.pdf
/content/drive/MyDrive/SOP/研究部/數據焦點_下週焦點-banner模板.pdf
/content/drive/MyDrive/SOP/研究部/數據焦點_每週焦點SOP_2022.pdf
/content/drive/MyDrive/SOP/研究部/數據焦點_財經日曆維護 SOP_2023.pdf
/content/drive/MyDrive/SOP/研究部/數據圖表管理_數據管理SOP_2023.pdf
/content/drive/MyDrive/SOP/研究部/數據圖表管理_個股正確數據覆蓋-2025.pdf
/content/drive/MyDrive/SOP/研究部/數據圖表管理_圖表批量更新模板-2024.pdf
/content/drive/MyDrive/SOP/研究部/數據圖表管理_數據批量上傳模板.pdf
/content/drive/MyDrive/SOP/研究部/數據批量更新模板-2024.pdf
/co

# 輸入Gemini API key（...Y-HE那把）

In [ ]:
from google import genai
from google.genai import types
import time
import getpass
client = genai.Client(api_key=getpass.getpass('🔑'))

🔑··········


# 重建向量資料庫

In [ ]:
# Delete the File Search store
file_search_store = client.file_search_stores.list()[0]
client.file_search_stores.delete(name=file_search_store.name, config=types.DeleteFileSearchStoreConfig(force=True))
print(f"Deleted store: {file_search_store.name}")

# Create the File Search store with an optional display name
file_search_store = client.file_search_stores.create(config={'display_name': 'MM SOP'})
print(f"Created store: {file_search_store.name}")

Deleted store: fileSearchStores/mm-sop-szq4h7lc9kil
Created store: fileSearchStores/mm-sop-xkf5u220mmzp


# 雲端硬碟SOP文件上傳向量資料庫

In [ ]:
import time
import uuid # Import uuid for generating unique temporary filenames
import shutil # Import shutil for file copy operations

for file_path in file_paths:
    file_name = os.path.basename(file_path)
    dir_name = os.path.dirname(file_path)

    # Get original file extension
    _, extension = os.path.splitext(file_name)

    # Generate a unique temporary filename for the copy
    temp_file_name = f"{uuid.uuid4().hex}{extension}"
    temp_file_path = os.path.join(dir_name, temp_file_name)

    print(f"檔案: {file_path}")
    print(f"副本: {temp_file_path}")

    try:
        # Step 1: Make a copy of the original file to a temporary safe file name
        shutil.copy2(file_path, temp_file_path)

        # Step 2: Upload the temporary file using the original file's (ASCII-safe) display name
        operation = client.file_search_stores.upload_to_file_search_store(
            file=temp_file_path, # Upload the temporary file copy
            file_search_store_name=file_search_store.name, # Use the existing file_search_store.name
            config={
                # 'display_name' : ascii_safe_display_name, # Use the ASCII-safe original name for display
                'display_name' : file_name, # Use the original name for display
            }
        )
        print(f'上傳副本')
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
    finally:
        # Step 3: Delete the temporary file copy
        if os.path.exists(temp_file_path):
            os.remove(temp_file_path)
            print('刪除副本')
        else:
            print(f"Temporary file '{temp_file_path}' not found to delete.")
        print()

# Wait until import is complete
while not operation.done:
    time.sleep(5)
    operation = client.operations.get(operation)
list(client.file_search_stores.list())

檔案: /content/drive/MyDrive/SOP/研究部/MM中國數據.docx
副本: /content/drive/MyDrive/SOP/研究部/82727174c97f4aca913fed329d2ee8d2.docx
上傳副本
刪除副本

檔案: /content/drive/MyDrive/SOP/研究部/產業決策平台SOP.docx
副本: /content/drive/MyDrive/SOP/研究部/1405b59de54e42abb19b43195abce45a.docx
上傳副本
刪除副本

檔案: /content/drive/MyDrive/SOP/研究部/月報檢查重點.docx
副本: /content/drive/MyDrive/SOP/研究部/a3e97e016cd54516af79086edf7127dc.docx
Error processing /content/drive/MyDrive/SOP/研究部/月報檢查重點.docx: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}
刪除副本

檔案: /content/drive/MyDrive/SOP/研究部/月報月曆 SOP.pdf
副本: /content/drive/MyDrive/SOP/研究部/0eb8441caf654bf4803ebbaeb2e34744.pdf
上傳副本
刪除副本

檔案: /content/drive/MyDrive/SOP/研究部/月報行情圖 SOP.pdf
副本: /content/drive/MyDrive/SOP/研究部/80e4714d48414b3d9b31bb93b4a69c9e.pdf
上傳副本
刪除副本

檔案: /content/drive/MyDrive/SOP/研究部/快報_語音朗讀 SOP_2023.pdf
副本: /content/drive/MyDrive/SOP/研究部/2fde69ca90504477b7fde2b9f1cd4c8b.pdf
上傳副本
刪除副本

檔案: /content/drive/MyDrive/SOP/研究部/快報_圖文編輯 

[FileSearchStore(
   active_documents_count=27,
   create_time=datetime.datetime(2025, 11, 19, 7, 49, 49, 542632, tzinfo=TzInfo(UTC)),
   display_name='MM SOP',
   name='fileSearchStores/mm-sop-xkf5u220mmzp',
   size_bytes=38150138,
   update_time=datetime.datetime(2025, 11, 19, 7, 49, 49, 542632, tzinfo=TzInfo(UTC))
 )]